# 第4回：深層学習による画像認識（姿勢推定・物体検出）

**このNotebookでできるようになること**

1. MediaPipe で人の **姿勢（33個の関節点）** を検出し，関節の角度を計算する
2. **YOLOv5** で画像・動画・カメラ映像から **80種類の物体** を検出する
3. 検出した物体を時刻とともに **CSVに記録** し，集計する
4. 自分たちで集めた画像で YOLO を学習させる手順（プロジェクト用）を知る

## 深層学習とは

大量の画像と正解ラベルから，画像の特徴の取り出し方そのものを学習する手法です．
第2回のHOGやHaarは人が設計した特徴を使いましたが，深層学習は特徴を自動で獲得するため，対象が複雑でも高い精度が出ます．
学習には大量の計算が必要ですが，**学習済みモデルを使う（推論）だけならノートPCのCPUで動きます**．

## 事前準備

`det/yolov5s.onnx` を用意してください（`image_recognition/README.md` 参照）．

In [ ]:
import os
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def set_japanese_font():
    # matplotlib のグラフ・タイトルの日本語が □（豆腐）にならないように日本語フォントを設定する
    # Windows ではフォントファイル（C:\Windows\Fonts）を直接登録するので，
    # matplotlib のフォントキャッシュが古くても・ユーザー単位でインストールしたフォントでも確実に使える
    import sys
    import matplotlib as mpl
    from matplotlib import font_manager as fm

    def use(name):
        mpl.rcParams["font.family"] = "sans-serif"
        mpl.rcParams["font.sans-serif"] = [name, "DejaVu Sans"]   # 日本語 → 英数字の順に探す
        mpl.rcParams["axes.unicode_minus"] = False                # 負号「−」の文字化け対策
        return name

    if sys.platform.startswith("win"):
        font_dirs = [os.path.join(os.environ.get("WINDIR", r"C:\Windows"), "Fonts"),
                     os.path.join(os.environ.get("LOCALAPPDATA", ""), "Microsoft", "Windows", "Fonts")]
        # 優先順：メイリオ → 游ゴシック → BIZ UDゴシック → MS ゴシック
        for fname in ["meiryo.ttc", "YuGothM.ttc", "YuGothR.ttc", "BIZ-UDGothicR.ttc", "msgothic.ttc"]:
            for d in font_dirs:
                path = os.path.join(d, fname)
                if os.path.exists(path):
                    try:
                        fm.fontManager.addfont(path)
                        return use(fm.FontProperties(fname=path).get_name())
                    except Exception:
                        pass

    # Mac / Linux，または上のファイルが見つからなかった場合はフォント名で探す
    available = {f.name for f in fm.fontManager.ttflist}
    for name in ["Meiryo", "Yu Gothic", "BIZ UDGothic", "MS Gothic", "Hiragino Sans",
                 "Hiragino Kaku Gothic ProN", "Noto Sans CJK JP", "Noto Sans JP", "IPAexGothic", "IPAGothic"]:
        if name in available:
            return use(name)

    print("日本語フォントが見つかりません．グラフの日本語が □ になる場合は，"
          "Windowsの［設定］→［時刻と言語］→［言語］で日本語の補助フォントを追加してください．")
    return None

print("matplotlib font:", set_japanese_font())

def show(img_bgr, title="", size=(8, 6)):
    plt.figure(figsize=size)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

## 1. MediaPipe Pose：姿勢推定と関節角度

MediaPipe Pose は身体の33個のランドマーク（肩・肘・手首・腰・膝・足首など）を検出します．
3点の座標から角度を計算すれば，「肘が曲がっているか」「しゃがんでいるか」などの **動作の数値化** ができます．

In [ ]:
# [SKIP-TEST]
import mediapipe as mp

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(static_image_mode=False, model_complexity=1,
                    min_detection_confidence=0.5, min_tracking_confidence=0.5)

def calculate_angle(a, b, c):
    # 3点 a-b-c のなす角（度）．b が頂点
    a, b, c = np.array(a), np.array(b), np.array(c)
    rad = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    ang = abs(rad * 180.0 / np.pi)
    return 360 - ang if ang > 180 else ang

JOINTS = {  # 表示名: (始点, 頂点, 終点)
    "L elbow": ("LEFT_SHOULDER", "LEFT_ELBOW", "LEFT_WRIST"),
    "R elbow": ("RIGHT_SHOULDER", "RIGHT_ELBOW", "RIGHT_WRIST"),
    "L knee": ("LEFT_HIP", "LEFT_KNEE", "LEFT_ANKLE"),
    "R knee": ("RIGHT_HIP", "RIGHT_KNEE", "RIGHT_ANKLE"),
}

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    results = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        h, w = frame.shape[:2]
        lm = results.pose_landmarks.landmark
        for name, (p1, p2, p3) in JOINTS.items():
            pts = [(lm[getattr(mp_pose.PoseLandmark, p)].x * w, lm[getattr(mp_pose.PoseLandmark, p)].y * h)
                   for p in (p1, p2, p3)]
            ang = calculate_angle(*pts)
            cv2.putText(frame, f"{name}: {int(ang)}", (int(pts[1][0]), int(pts[1][1])),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    cv2.putText(frame, "Press 'q' to quit", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.imshow("MediaPipe Pose", frame)
    if cv2.waitKey(10) & 0xFF == ord("q"):
        break

pose.close()
cap.release()
cv2.destroyAllWindows()

> 手の21点ランドマークを3Dで表示する `scripts/hand_gesture.py` も用意しています（`python scripts/hand_gesture.py`）．

## 2. YOLOv5 による物体検出

**YOLO (You Only Look Once)** は画像を1回ネットワークに通すだけで，複数の物体の位置（矩形）と種類を同時に出力する物体検出モデルです．
ここでは軽量な **YOLOv5s** を ONNX 形式に変換したものを，OpenCV の DNN モジュールで動かします（PyTorch不要）．

出力は「候補となる矩形の一覧」で，各行に `[中心x, 中心y, 幅, 高さ, 物体らしさ, クラス1のスコア, …, クラス80のスコア]` が入っています．
信頼度の低い候補を捨て，重なった候補を **NMS（非最大抑制）** でまとめると最終結果になります．

In [ ]:
MODEL = "det/yolov5s.onnx"
NAMES = "det/coco.names"

INPUT_SIZE = 640
CONF_THRESHOLD = 0.45      # 物体らしさのしきい値
SCORE_THRESHOLD = 0.5      # クラススコアのしきい値
NMS_THRESHOLD = 0.45       # 重なった矩形をまとめる度合い

classes = open(NAMES, encoding="utf-8").read().strip().split("\n")
print("クラス数:", len(classes), classes[:10])

if os.path.exists(MODEL):
    net = cv2.dnn.readNet(MODEL)
    print("モデルを読み込みました:", MODEL)
else:
    net = None
    print("det/yolov5s.onnx がありません．README の手順で用意してください．")

In [ ]:
def detect_objects(net, img):
    # 画像(BGR) → [(クラス名, 信頼度, (x, y, w, h)), ...]
    h, w = img.shape[:2]
    blob = cv2.dnn.blobFromImage(img, 1 / 255.0, (INPUT_SIZE, INPUT_SIZE), swapRB=True, crop=False)
    net.setInput(blob)
    out = net.forward(net.getUnconnectedOutLayersNames())[0][0]   # 形: (候補数, 85)
    boxes, confs, ids = [], [], []
    for row in out:
        conf = float(row[4])
        if conf < CONF_THRESHOLD:
            continue
        scores = row[5:]
        cid = int(np.argmax(scores))
        if scores[cid] < SCORE_THRESHOLD:
            continue
        cx, cy, bw, bh = row[:4]
        boxes.append([int((cx - bw / 2) * w / INPUT_SIZE), int((cy - bh / 2) * h / INPUT_SIZE),
                      int(bw * w / INPUT_SIZE), int(bh * h / INPUT_SIZE)])
        confs.append(conf)
        ids.append(cid)
    keep = cv2.dnn.NMSBoxes(boxes, confs, CONF_THRESHOLD, NMS_THRESHOLD)
    keep = [int(i) for i in np.array(keep).ravel()] if len(keep) else []
    return [(classes[ids[i]], confs[i], tuple(boxes[i])) for i in keep]


def draw_detections(img, detections):
    for name, conf, (x, y, w, h) in detections:
        cv2.rectangle(img, (x, y), (x + w, y + h), (255, 178, 50), 2)
        label = f"{name}: {conf:.2f}"
        (tw, th), base = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
        cv2.rectangle(img, (x, y), (x + tw, y + th + base), (0, 0, 0), cv2.FILLED)
        cv2.putText(img, label, (x, y + th), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 1, cv2.LINE_AA)
    return img

### 2-1. 画像1枚で試す

In [ ]:
if net is not None:
    img = cv2.imread("data/img/img01.jpg")
    t0 = time.time()
    dets = detect_objects(net, img)
    print(f"推論時間: {time.time() - t0:.2f} 秒  検出数: {len(dets)}")
    for name, conf, box in dets:
        print(f"  {name:12s} {conf:.2f} {box}")
    show(draw_detections(img, dets), "YOLOv5s 物体検出")

### 2-2. 動画に対して実行し，検出した物体をCSVに記録する

第3回と同じ考え方で，**「何秒に，何が，いくつ」** を記録します．

In [ ]:
if net is not None:
    cap = cv2.VideoCapture("data/vtest.avi")
    fps = cap.get(cv2.CAP_PROP_FPS)
    records, num = [], 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if num % 15 == 0:                                   # 15フレームごと
            for name, conf, _ in detect_objects(net, frame):
                records.append({"time": round(num / fps, 2), "object": name, "conf": round(conf, 2)})
        num += 1
    cap.release()
    det_df = pd.DataFrame(records)
    det_df.to_csv("data/yolo_detections.csv", index=False, encoding="utf-8-sig")
    print("記録数:", len(det_df))
    det_df.head()

In [ ]:
if net is not None and len(det_df):
    # 物体の種類ごとの検出数
    det_df["object"].value_counts().plot(kind="barh", figsize=(6, 4))
    plt.title("検出された物体の種類と数")
    plt.xlabel("検出数")
    plt.tight_layout()
    plt.show()

    # 時刻ごとの人数（YOLOによる）
    people = det_df[det_df["object"] == "person"].groupby("time").size()
    plt.figure(figsize=(10, 4))
    plt.plot(people.index, people.values)
    plt.xlabel("時間（秒）"); plt.ylabel("person の検出数")
    plt.title("YOLOv5 による人数の推移（第3回のHOGと比べてみよう）")
    plt.grid(alpha=0.3)
    plt.show()

### 2-3. Webカメラでリアルタイム検出

同じ処理を `scripts/yolo_detect.py` にまとめてあります．Anaconda Prompt から次のように実行できます．

```bash
python scripts/yolo_detect.py --source 0                      # Webカメラ
python scripts/yolo_detect.py --source data/vtest.avi --csv data/yolo.csv   # 動画ファイル＋CSV記録
python scripts/yolo_detect.py --source data/img/img01.jpg     # 画像
python scripts/yolo_detect.py --model det/best.onnx --names det/my.names --source 0   # 自作モデル
```

In [ ]:
# [SKIP-TEST]
if net is not None:
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        dets = detect_objects(net, frame)
        cv2.imshow("YOLOv5 Object Detection", draw_detections(frame, dets))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()

## 3. 自分たちの物体を検出させるには（プロジェクト用）

YOLOv5s が検出できるのは COCO データセットの80種類だけです．「ヘルメット」「ゴミの種類」のような **独自の物体** を検出するには，自分たちで画像を集めて学習させます．
手順は `advanced/yolo_custom_training.md` にまとめています．概要は次のとおりです．

1. **画像を集める**（各クラス 200〜500枚が目安．`advanced/google_image_search.py` や自分で撮影）
2. **アノテーション**（画像の中の物体を矩形で囲んでラベルを付ける．ブラウザで使える無料ツールを紹介）
3. **Google Colab（無料GPU）で学習**（YOLOv5 の `train.py`．数十分〜数時間）
4. **ONNX に変換**（`export.py`）して `det/` に置く
5. このNotebook・`scripts/yolo_detect.py` で **クラス名ファイルを差し替えて** 動かす

昨年度のプロジェクト例：「ゴミ分類モデル」（缶・瓶・紙・プラを検出）「自転車の危険運転検知」（自転車利用者とヘルメットを検出）はどちらもこの手順で作られました．

## まとめ：3回の演習で手に入れた道具

| 段階 | 道具 |
|---|---|
| 第2回 データ取得 | 画像・動画・カメラの入出力，人物・顔検出，顔の向き，タイムラプス |
| 第3回 加工・可視化 | 検出結果の表・時系列グラフ，移動平均，変化点検出，CSV保存 |
| 第4回 深層学習 | MediaPipe 姿勢推定（関節角度），YOLOv5 物体検出（80種），検出結果のCSV記録，独自モデルの学習手順 |

**プロジェクトの考え方**：「どこで・何を・どのように数えれば，誰のどんな困りごとが減るか」を先に決め，その問いに答えるために上の道具を組み合わせてください．SNS分析テーマの道具（投稿画像への物体検出）と組み合わせることもできます．